# 04 — Product Performance
Top/bottom products by revenue and profit; quantity sold; category mix.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Top 10 Products by Revenue ───────────────────────────────────────────────
top_rev = fetch_df(cur, """
    SELECT p.product_name, p.category,
           ROUND(SUM(oi.sales),2)  AS revenue,
           ROUND(SUM(oi.profit),2) AS profit,
           SUM(oi.quantity)        AS units_sold
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_name, p.category
    ORDER BY revenue DESC LIMIT 10
""")

cat_palette = {"Furniture": PALETTE[0], "Office Supplies": PALETTE[1],
               "Technology": PALETTE[2]}
colors = [cat_palette.get(c, PALETTE[3]) for c in top_rev["category"][::-1]]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_rev["product_name"].str[:42][::-1],
               top_rev["revenue"][::-1],
               color=colors, edgecolor="white")
for bar in bars:
    ax.text(bar.get_width() + 200,
            bar.get_y() + bar.get_height()/2,
            fmt_usd(bar.get_width()),
            va="center", fontsize=8)

handles = [mpatches.Patch(color=v, label=k) for k, v in cat_palette.items()]
ax.legend(handles=handles, frameon=False, fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Revenue (USD)")
ax.set_title("Top 10 Products by Revenue")
plt.tight_layout()
plt.show()


In [ ]:
# ── Top 10 Products by Profit ────────────────────────────────────────────────
top_prof = fetch_df(cur, """
    SELECT p.product_name, p.sub_category,
           ROUND(SUM(oi.profit),2) AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_name, p.sub_category
    ORDER BY profit DESC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_prof["product_name"].str[:42][::-1],
               top_prof["profit"][::-1],
               color=PALETTE[1], edgecolor="white")
for bar, margin in zip(bars, top_prof["margin_pct"][::-1]):
    ax.text(bar.get_width() + 30,
            bar.get_y() + bar.get_height()/2,
            f"{fmt_usd(bar.get_width())} ({margin}%)",
            va="center", fontsize=8)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Profit (USD)")
ax.set_title("Top 10 Products by Profit")
plt.tight_layout()
plt.show()


In [ ]:
# ── Bottom 10 Products (Loss-Makers) ─────────────────────────────────────────
bot_prof = fetch_df(cur, """
    SELECT p.product_name, p.sub_category,
           ROUND(SUM(oi.profit),2) AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM order_items oi JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_name, p.sub_category
    ORDER BY profit ASC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(bot_prof["product_name"].str[:42],
               bot_prof["profit"],
               color=PALETTE[3], edgecolor="white")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
for bar, margin in zip(bars, bot_prof["margin_pct"]):
    ax.text(bar.get_width() - 30,
            bar.get_y() + bar.get_height()/2,
            f"{fmt_usd(bar.get_width())} ({margin}%)",
            va="center", ha="right", fontsize=8,
            color="white", fontweight="bold")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Profit (USD)")
ax.set_title("Bottom 10 Loss-Making Products")
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
